# Lab 4.2 - Code Refactoring with Gemini API
Using Google Generative AI (Gemini) to refactor Python code

In [4]:
# Cell 1: Import Libraries
import os
import inspect
from dotenv import load_dotenv
import google.generativeai as genai

In [5]:
# Cell 2: Initialize Model and Configuration
# Load API Key from .env file
load_dotenv(os.path.join(os.path.dirname(os.path.abspath('.')), '.env'))
api_key = os.getenv("GEMINI_API_KEY")

# Configure Gemini API
genai.configure(api_key=api_key)

# Initialize model
generation_config = genai.types.GenerationConfig(
    temperature=0.2
)

model = genai.GenerativeModel(
    'gemini-2.5-flash',
    generation_config=generation_config
)

# Define prompts
SURROUND = ("You will be provided with a Python function enclosed with {{{ CODE }}} and a desired code change enclosed "
            "with {{{ CHANGE }}}.")
SINGLE_TASK = "Your task is to return a new Python function with the required change."

def get_estimated_user_costs(prompts_data):
    """Calculate estimated costs for user prompts"""
    costs = {}
    for entry in prompts_data:
        user = entry["user"]
        prompt = entry["prompt"]
        estimated_tokens = len(prompt) / 4  # 4 is the average number of tokens per character
        cost = estimated_tokens * 0.0001  # 0.0001 is the cost per token

        if user not in costs:
            costs[user] = cost
        else:
            costs[user] += cost

    return costs

def get_user_prompt(func: callable, change: str) -> str:
    """Generate user prompt for code refactoring"""
    return f"""
    CHANGE: {{{{ {change} }}}}
    
    CODE: {{{{ {inspect.getsource(func)} }}}}
    
    REFACTORED CODE:
    """

In [6]:
# Cell 3: Execute and Print Results
changes = [
    "Replace dictionary key-value assignments with calls to dict.get()",
    "Convert hard-coded integers to global constants",
    "Replace dictionary key-value assignments with calls to dict.get() and convert hard-coded integers to global constants",
]

for change in changes:
    user_prompt = get_user_prompt(get_estimated_user_costs, change)
    full_prompt = f"{SURROUND} {SINGLE_TASK}\n{user_prompt}"

    response = model.generate_content(full_prompt)

    print(f"Change: {change}")
    print("-" * 80)
    print(response.text)
    print("\n" + "=" * 80 + "\n")

Change: Replace dictionary key-value assignments with calls to dict.get()
--------------------------------------------------------------------------------
```python
def get_estimated_user_costs(prompts_data):
    """Calculate estimated costs for user prompts"""
    costs = {}
    for entry in prompts_data:
        user = entry["user"]
        prompt = entry["prompt"]
        estimated_tokens = len(prompt) / 4  # 4 is the average number of tokens per character
        cost = estimated_tokens * 0.0001  # 0.0001 is the cost per token

        costs[user] = costs.get(user, 0) + cost

    return costs
```


Change: Convert hard-coded integers to global constants
--------------------------------------------------------------------------------
```python
TOKENS_PER_CHARACTER_AVERAGE = 4
COST_PER_TOKEN = 0.0001

def get_estimated_user_costs(prompts_data):
    """Calculate estimated costs for user prompts"""
    costs = {}
    for entry in prompts_data:
        user = entry["user"]
        promp